# Advanced exercise

Try creating a 3-way, perhaps bringing Gemini into the conversation! One student has completed this - see the implementation in the community-contributions folder.

The most reliable way to do this involves thinking a bit differently about your prompts: just 1 system prompt and 1 user prompt each time, and in the user prompt list the full conversation so far.

Something like:

```python
system_prompt = """
You are Alex, a chatbot who is very argumentative; you disagree with anything in the conversation and you challenge everything, in a snarky way.
You are in a conversation with Blake and Charlie.
"""

user_prompt = f"""
You are Alex, in conversation with Blake and Charlie.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Alex.
"""
```


In [ ]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
grok_api_key = os.getenv("GROK_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [ ]:
# Constants
gemini_model = "gemma3:270m"
gpt_model = "gpt-oss:20b"
ollama_model = "llama3.1:8b"


In [ ]:
gpt_system_prompt = (
    "You are Alex. "
    "You are very argumentative and confrontational. "
    "You disagree with what the other people say and challenge "
    "their opinions, assumptions, and reasoning. Be snarky and "
    "sarcastic when appropriate. "
    "You are in a conversation with Blake and Charlie."
)

ollama_system_prompt = (
    "You are Blake."
    "You are very polite, courteous, and agreeable. "
    "You try to agree with what the other people say or find "
    "common ground. If someone is argumentative, remain calm, "
    "de-escalate the conversation, and keep chatting."
    "You are in a conversation with Alex and Charlie."
)

gemini_system_prompt = (
    "You are Charlie."
    "You are very argumentative and skeptical. "
    "You challenge what the other people say and question "
    "their opinions, assumptions, and reasoning. "
    "Push back on their arguments in a direct and confident way, "
    "with some sarcasm when appropriate."
    "You are in a conversation with Alex and Blake."
)


In [ ]:
def call_gemini_local(conversation):
    gemini_user_prompt = (
        f"You are Charlie, in a conversation with Alex and Blake.\n\n"
        f"The conversation so far is as follows:\n\n"
        f"{conversation}\n\n"
        f"Now, respond with what you would like to say next, as Charlie."
    )
    messages = [
        {"role": "system", "content": gemini_system_prompt},
        {"role": "user", "content": gemini_user_prompt},
    ]

    response = ollama.chat.completions.create(model=gemini_model, messages=messages)
    return response.choices[0].message.content


def call_ollama_local(conversation):
    ollama_user_prompt = (
        f"You are Blake, in a conversation with Alex and Charlie.\n\n"
        f"The conversation so far is as follows:\n\n"
        f"{conversation}\n\n"
        f"Now, respond with what you would like to say next, as Blake."
    )

    messages = [
        {"role": "system", "content": ollama_system_prompt},
        {"role": "user", "content": ollama_user_prompt},
    ]

    # print("ollama messages", messages)
    response = ollama.chat.completions.create(model=ollama_model, messages=messages)
    return response.choices[0].message.content


def call_gpt_local(conversation):
    gpt_user_prompt = (
        f"You are Alex, in a conversation with Blake and Charlie.\n\n"
        f"The conversation so far is as follows:\n\n"
        f"{conversation}\n\n"
        f"Now, respond with what you would like to say next, as Alex."
    )
    messages = [
        {"role": "system", "content": gpt_system_prompt},
        {"role": "user", "content": gpt_user_prompt},
    ]

    response = ollama.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
# gpt_messages = "Hi "
# ollama_messages = "Hi there"
# gemini_messages = "Hi everyone"


# conversation = (
#     f"Alex : {gpt_messages}\n"
#     f"Blake : {ollama_messages}\n"
#     f"Charlie : {gemini_messages}"
# )

conversation = "What is 2 + 2?"
display(Markdown(f"### Conversation so far :\n{conversation}\n"))

print("--------------- Conversation Started ----------------- ")

for i in range(5):
    # Alex
    print(conversation)
    gpt_next = call_gpt_local(conversation=conversation)
    display(Markdown(f"### Alex:\n{gpt_next}\n"))
    conversation += f"\nAlex : {gpt_next}"

    # Blake
    ollama_text = call_ollama_local(conversation=conversation)
    display(Markdown(f"### Blake:\n{ollama_text}\n"))
    conversation += f"\nBlake : {ollama_text}"

    # Charlie
    gemini_text = call_gemini_local(conversation=conversation)
    display(Markdown(f"### Charlie:\n{gemini_text}\n"))
    conversation += f"\nCharlie : {gemini_text}"
    print(f"------ Iteration {i + 1} Ended ------ ")

print("--------------- Conversation Ended ----------------- ")